## Clean DB Builder (Brawl Stars Ranked)

This notebook orchestrates a safe, repeatable pull that writes a clean, analysis-ready SQLite database under `data_clean/`.

- Reads `BS_API_KEY` from the environment (no secrets stored).
- Writes the clean `matches` table; raw copy is optional and off by default.
- Defaults minimize API calls (`fetch_player_data=False`).


### Safety

- Ensure your current machine IP is allow-listed for the API key.
- The notebook only creates files under `data_clean/` (and optional raw under `data_raw/` if enabled).
- Do not print or hardcode secrets.


In [ ]:
# Imports
import os
import json
import sqlite3
from datetime import datetime, timezone
import pandas as pd

import sys, os
project_root = os.path.abspath("..")  # from notebooks/ to project root
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.env_utils import ensure_directories, get_api_key
from DB_Data_Pull.pull_data_4 import process_tags_and_write_async
from data_clean import write_season_metadata, compute_season_metadata


### Config

Set the season label and run parameters. `fetch_player_data=False` by default to minimize API calls. Update timestamps each season.


In [ ]:
# Season + runtime
season_label = "season49"  # e.g., 'season37'
season_version = "1"
latest_runtime = datetime(2026, 4, 15, tzinfo=timezone.utc)  # ingest sets strictly after this time

# Paths
clean_db_path = f"{project_root}/data_clean/{season_label}/v{season_version}_clean.db"


### Environment and directories

Creates required folders and loads the API key from `BS_API_KEY` (falls back to `BRAWLSTARS_API_KEY`).


In [ ]:
# The API key is read from the environment. Nothing is stored in the repo.
#   export BS_API_KEY="..."      (see .env.example)
from utils.env_utils import get_api_key

API_KEY = get_api_key()
print("API key loaded from environment:", bool(API_KEY))


## Single Run

### Seed tags

Choose how to seed the crawl. Option A draws from an existing raw DB (if available). Option B uses a manual list.


In [ ]:
from utils.seeding import sample_seed_tags_from_clean_db

USE_RANDOM_SEEDS = False  # True to sample from the clean DB

if USE_RANDOM_SEEDS:
    try:
        initial_tags = sample_seed_tags_from_clean_db(
            clean_db_path=clean_db_path,
            num_tags=20,
            elo_range=(10, 23),  # optional; remove to sample across all elo
        )
        if not initial_tags:
            raise RuntimeError("No tags found from clean DB sampling.")
    except Exception as e:
        print("Falling back to manual seeds due to error:", e)
        initial_tags = ["#9UUU9QVU"]
else:
    initial_tags = ["#9UUU9QVU", "QLCJGQUP", "9299U02V0"]  # replace or expand

print(f"Seed tags: {len(initial_tags)}")
print(initial_tags[:5])


### Pull: build clean DB - Single Run

Runs the async pipeline to write `matches` into the clean DB. Raw writing remains disabled unless `write_raw_copy=True`.


In [ ]:
# Crawler knobs
max_depth = 2
batch_size = 1500
concurrency = 30
fetch_player_data = False  # toggle True to enrich player profiles

# elo range for queueing tags
elo_queue_min = 14  
elo_queue_max = 23

# elo range for keeping games in the DB
elo_game_min = 10
elo_game_max = 23

# skip tags fetched within this window (0 = disabled)
fetched_tags_ttl_hours = 24.0

# flush rows to DB every N BFS batches to cap memory use (0 = disabled)
flush_every_n_batches = 0

async def _run_pull():
    await process_tags_and_write_async(
        player_tags=initial_tags,
        api_key=API_KEY,
        latest_runtime=latest_runtime,
        max_depth=max_depth,
        batch_size=batch_size,
        concurrency=concurrency,            # parallelism
        fetch_player_data=fetch_player_data,
        clean_db_path=clean_db_path,
        prefilter_initial_tags=False,           # optionally prefilter seed tags by elo_queue_*
        elo_queue_min=elo_queue_min,
        elo_queue_max=elo_queue_max,
        elo_game_min=elo_game_min,
        elo_game_max=elo_game_max,
        requests_per_second=1000.0,               # smooth send rate (not the same as concurrency)
        fetched_tags_ttl_hours=fetched_tags_ttl_hours,
        flush_every_n_batches=flush_every_n_batches,
    )

await _run_pull()

## Queued Runs

### Seed Tags to External File

In [ ]:
from utils.seeding import sample_seed_tags_from_clean_db

# Prepare: absolute paths and seed tags file
seed_tags_file = f"{project_root}/data_clean/{season_label}/seed_tags.txt"
seed_db = clean_db_path
# seed_db = f"{project_root}/data_clean/season42/v1_clean.db"

# Option A: manual tags
# initial_tags = ["#9UUU9QVU", "QLCJGQUP", "9299U02V0"]

# Option B: sample from clean DB if it already has rows
initial_tags = sample_seed_tags_from_clean_db(
    clean_db_path=seed_db,
    num_tags=500,
    elo_range=(15, 23),  # optional
)

# Normalize and dedupe
tags = []
seen = set()
for t in initial_tags:
    t = t if t.startswith("#") else f"#{t}"
    if t not in seen:
        seen.add(t)
        tags.append(t)

# Write to file (one per line)
os.makedirs(os.path.dirname(seed_tags_file), exist_ok=True)
with open(seed_tags_file, "w") as f:
    for t in tags:
        f.write(t + "\n")

print("Wrote", len(tags), "tags to", seed_tags_file)
print("Clean DB:", clean_db_path)

### Pull Tags: Build Clean DB with Queued Small Runs

In [ ]:
# Queue many short runs using subprocesses (recommended)
import shlex
import subprocess

# Per-run size: keep small to bound memory
per_run_tags = 5 # number of starting tags per run, 5 for depth 2
max_runs = 500  # optional cap, will stop even if more tags remain
sleep_between_runs = 2.0

# Crawler knobs
max_depth = 2
batch_size = 1500
concurrency = 30
fetch_player_data = False  # toggle True to enrich player profiles
prefilter_initial_tags = False

# elo range for queueing tags
elo_queue_min = 13 
elo_queue_max = 23

# elo range for keeping games in the DB
elo_game_min = 12
elo_game_max = 23

# skip tags fetched within this window (0 = disabled)
fetched_tags_ttl_hours = 24.0

# flush rows to DB every N BFS batches to cap memory use (0 = disabled)
flush_every_n_batches = 0

# Build the command; run from project root so `-m scripts.queue_runs` resolves
cmd = [
    "python3", "-m", "scripts.queue_runs",
    "--tags-file", seed_tags_file,
    "--per-run-tags", str(per_run_tags),
    "--max-runs", str(max_runs),
    "--sleep-between-runs", str(sleep_between_runs),
    "--clean-db-path", clean_db_path,
    "--latest-runtime", str(latest_runtime),
    "--max-depth", str(max_depth),
    "--batch-size", str(batch_size),
    "--concurrency", str(concurrency),
    "--requests-per-second", "1000.0",
    "--elo-game-min", str(elo_game_min),
    "--elo-game-max", str(elo_game_max),
    "--fetched-tags-ttl-hours", str(fetched_tags_ttl_hours),
    "--flush-every-n-batches", str(flush_every_n_batches),
    # "--fetch-player-data",            # do not pass value, uncomment for "True"
    # "--prefilter-initial-tags",       # do not pass value, uncomment for "True"
]

print("Running:\n", " ".join(shlex.quote(c) for c in cmd))
# IMPORTANT: set cwd to project root so module resolution works
subprocess.run(cmd, check=True, cwd=project_root)

# Elo Normalization

In [ ]:
# Compute and write time-local ECDF skill features (fixed bins, no merging)
import sys, subprocess, sqlite3
from pathlib import Path

# Ensure the repo root is importable
PROJECT_ROOT = Path("/Users/elifried/Desktop/Brawl Stars/BrawlStars_ETL").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Optional explicit season label (None = auto-derive from DB path)
SEASON = "season48"
# Required: path to the clean DB to modify (in-place)
CLEAN_DB_PATH = f"{project_root}/data_clean/{SEASON}/v1_clean.db"

# Hyperparameters
BIN_WIDTH_DAYS = 3          # fixed bin width (days); no merging
MIN_BIN_COUNT = 100000        # minimum samples per bin for coverage_ok=1
EPSILON = 1e-3              # clipping for percentile-to-score mapping
MAPPING = "logit"          # "normal" or "logit"
FALLBACK_STRATEGY = "none"  # "none" or "global_season_ecdf"

# Optional performance knobs
READ_BATCH_SIZE = 200_000
UPDATE_BATCH_SIZE = 20_000

cmd = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "compute_skill_features.py"),
    "--clean-db-path", str(CLEAN_DB_PATH),
    "--bin-width-days", str(BIN_WIDTH_DAYS),
    "--min-bin-count", str(MIN_BIN_COUNT),
    "--epsilon", str(EPSILON),
    "--mapping", MAPPING,
    "--fallback-strategy", FALLBACK_STRATEGY,
    "--read-batch-size", str(READ_BATCH_SIZE),
    "--update-batch-size", str(UPDATE_BATCH_SIZE),
]
if SEASON:
    cmd += ["--season", str(SEASON)]

print("Running:", " ".join([str(x) for x in cmd]))
subprocess.run(cmd, check=True)

# Optional: quick verification summary
VERIFY = True
if VERIFY:
    con = sqlite3.connect(str(CLEAN_DB_PATH))
    try:
        total = con.execute("SELECT COUNT(*) FROM matches").fetchone()[0]
        nn = con.execute(f"SELECT COUNT(*) FROM matches WHERE { 'skill_ns' } IS NOT NULL").fetchone()[0]
        ok1 = con.execute(f"SELECT COUNT(*) FROM matches WHERE { 'skill_ns_ok' } = 1").fetchone()[0]
        ok0 = con.execute(f"SELECT COUNT(*) FROM matches WHERE { 'skill_ns_ok' } = 0").fetchone()[0]
        print(f"Rows total: {total:,}")
        print(f"skill_ns not null: {nn:,}")
        print(f"coverage_ok=1: {ok1:,} | coverage_ok=0: {ok0:,}")
        
        # Time-bin summary (buckets, counts, date ranges)
        cur = con.execute(
            """
            SELECT bin_start_utc, bin_end_utc, n_samples, coverage_ok
            FROM skill_bin_metadata
            WHERE season = ?
            ORDER BY bin_start_utc
            """,
            (SEASON,),
        )
        bins = cur.fetchall()
        print(f"Time bins (width={BIN_WIDTH_DAYS} days): {len(bins)}")
        for idx, (b_start, b_end, n_samples, ok) in enumerate(bins, 1):
            status = "OK" if int(ok) == 1 else "LOW"
            print(f"{idx:>3}. {b_start} → {b_end} | n={n_samples:,} | {status}")
    finally:
        con.close()

# Data Summary

### Season metadata

Write a JSON sidecar summarizing the dataset.


In [ ]:
data_label = f"{season_label}_v{season_version}"

meta_path, meta_data = write_season_metadata(clean_db_path, data_label)
print("Metadata written to:", meta_path)
print(json.dumps(meta_data, indent=2))


### Quick validations

Sanity checks on counts, date ranges, and sample rows.


In [ ]:
con = sqlite3.connect(clean_db_path)
cur = con.cursor()

# Counts
num_matches = cur.execute("SELECT COUNT(*) FROM matches").fetchone()[0]
start_time = cur.execute("SELECT MIN(battle_time) FROM matches").fetchone()[0]
end_time = cur.execute("SELECT MAX(battle_time) FROM matches").fetchone()[0]
modes = cur.execute("SELECT COUNT(DISTINCT mode) FROM matches").fetchone()[0]
maps = cur.execute("SELECT COUNT(DISTINCT map) FROM matches").fetchone()[0]
print({"num_matches": num_matches, "start_time": start_time, "end_time": end_time, "unique_modes": modes, "unique_maps": maps})

# Head - Display first N rows as DataFrame
n_rows = 10
query = "SELECT id, battle_time, mode, map, record, star_brawler, star_power, star_player_tag, star_elo, avg_elo FROM matches ORDER BY battle_time DESC LIMIT ?"
df_head = pd.read_sql_query(query, con, params=(n_rows,))

# Display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
display(df_head)

con.close()
